# Training Recipe

The goal here is to have a full training recipe, from dataset loaders to model definition to the full training loop.

## Imports and Hyperparameters

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
EPOCHS = 10
D_MODEL = 128
D_HIDDEN = D_MODEL * 4      # typical FFN expansion ratio (4x)
D_OUT = 10                  # e.g. 10-class classification
SEQ_LEN = 64

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

## Dataset Loader

In [2]:
class RandomDataset(Dataset):
    def __init__(self, num_samples, seq_len, d_model, num_classes):
        self.x = torch.randn(num_samples, seq_len, d_model)
        self.y = torch.randint(0, num_classes, (num_samples,))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_dataset = RandomDataset(1000, SEQ_LEN, D_MODEL, D_OUT)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [5]:
for x_batch, y_batch in train_dataloader:
    print(x_batch.shape, y_batch.shape)
    break

torch.Size([32, 64, 128]) torch.Size([32])


In [6]:
x_batch, y_batch = next(iter(train_dataloader))
print(x_batch.shape, y_batch.shape)

torch.Size([32, 64, 128]) torch.Size([32])


## Building the Models

In [ ]:
# Note: We didn't end up using this in the models below.
# Add a SwiGLU module as PyTorch doesn't provide this out of the box.
class SwiGLU(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.w1 = nn.Linear(d_in, d_hidden, bias=False)
        self.v = nn.Linear(d_in, d_hidden, bias=False)
        self.w2 = nn.Linear(d_hidden, d_in, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))

In [ ]:
# Variant 1: Simple MLP with flatten at the start.
model_simple = nn.Sequential(
    nn.Flatten(start_dim=1),        # (B, T, C) --> (B, T*C)
    nn.Linear(SEQ_LEN*D_MODEL, D_HIDDEN),
    nn.SiLU(),
    nn.Linear(D_HIDDEN, D_HIDDEN),
    nn.SiLU(),
    nn.Linear(D_HIDDEN, D_OUT)
).to(DEVICE)

In [9]:
# Variant 2: With meanpooling.
class MeanPool(nn.Module):
    """Collapse sequence dim by averaging."""
    def forward(self, x):
        return x.mean(dim=1)  # (B, T, D) -> (B, D)

model_with_swiglu = nn.Sequential(
    nn.Linear(D_MODEL, D_HIDDEN),
    nn.ReLU(),
    nn.Linear(D_HIDDEN, D_HIDDEN),
    nn.ReLU(),
    MeanPool(),
    nn.Linear(D_HIDDEN, D_OUT)
).to(DEVICE)

## Training loop with loss and optim

In [10]:
# Define the model, loss and optim
model = model_simple
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)